In [3]:
import os
from typing import Literal
from dotenv import load_dotenv
from langchain_huggingface import ChatHuggingFace, HuggingFacePipeline
from langchain_openrouter import ChatOpenRouter
from tavily import TavilyClient
from deepagents import create_deep_agent
from deepagents.backends import FilesystemBackend

load_dotenv()

hf_token = os.getenv("HUGGINGFACEHUB_API_TOKEN")
hf_model = os.getenv("HUGGINGFACE_MODEL", "google/gemma-4-E2B-it")
tavily_api_key = os.getenv("TAVILY_API_KEY")
tavily = TavilyClient(api_key=tavily_api_key) if tavily_api_key else None

def internet_search(
    query: str,
    max_results: int = 5,
    topic: Literal["general", "news", "finance"] = "general",
):
    """
    Use this tool to make an internet search by providing a query.
    """
    print("PERFORMING A SEARCH ON: "+query)
    if not tavily:
        return "Tavily API key not found. Please set TAVILY_API_KEY in .env."
    return tavily.search(query, max_results=max_results, topic=topic)





hf_llm = HuggingFacePipeline.from_model_id(
    model_id=hf_model,
    task="text-generation"
)

  
model = ChatHuggingFace(llm=hf_llm)


agent = create_deep_agent(
    model=model,
    system_prompt="""You are a generic Deep Agent, an expert orchestrator designed to perform any task.
Your primary goal is to use the provided skill library to handle specialized requirements on-demand.
...""",
    tools=[internet_search],
    name="generic_deep_agent",
)



Loading weights: 100%|██████████| 1951/1951 [00:00<00:00, 5308.57it/s]


In [4]:
result = agent.invoke({"messages": [{"role": "user", "content": "Perform an internet search and find out who the current mayor of the city of Berne is"}]})
print(result["messages"][-1].content)

[transformers] Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


<bos><|turn>system
[{'type': 'text', 'text': 'You are a generic Deep Agent, an expert orchestrator designed to perform any task.\nYour primary goal is to use the provided skill library to handle specialized requirements on-demand.\n...\n\nYou are a Deep Agent, an AI assistant that helps users accomplish tasks using tools. You respond with text and tool calls. The user can see your responses and tool outputs in real time.\n\n## Core Behavior\n\n- Be concise and direct. Don\'t over-explain unless asked.\n- NEVER add unnecessary preamble ("Sure!", "Great question!", "I\'ll now...").\n- Don\'t say "I\'ll now do X" — just do it.\n- If the request is underspecified, ask only the minimum followup needed to take the next useful action.\n- If asked how to approach something, explain first, then act.\n\n## Professional Objectivity\n\n- Prioritize accuracy over validating the user\'s beliefs\n- Disagree respectfully when the user is incorrect\n- Avoid unnecessary superlatives, praise, or emotiona